# Distributed Training Pipeline Basics

This notebook is an interview-oriented PyTorch reference for building a training pipeline that remains observable and recoverable as it scales.

Topics:

- streaming data with deterministic rank and worker sharding
- `DataLoader` worker parallelism versus DistributedDataParallel (DDP)
- globally consistent optimizer steps and metrics
- atomic checkpoints with model, optimizer, scheduler, and RNG state
- restart behavior after worker or node failure
- data-consistency checks and debugging metrics

All executable cells run in one CPU process. The same components can be launched with `torchrun` for multi-GPU training.

## 1. Mental model

```text
object store / files / queue
            |
      streaming dataset
            |
   rank shard + worker shard       one process per accelerator
            |                     +--------------------------+
      DataLoader workers -------> | rank 0: model replica    |
                                  | rank 1: model replica    | -- all-reduce gradients
                                  | ...                      |
                                  +--------------------------+
                                              |
                              optimizer / scheduler step
                                              |
                           reduced metrics + atomic checkpoint
```

**Do not confuse the two forms of parallelism:**

| Mechanism | Purpose | State communication |
|---|---|---|
| `DataLoader(num_workers=N)` | parallelize reading, decoding, and preprocessing inside one rank | workers send batches to their parent rank |
| DDP with `world_size=N` | replicate the model and split data across training processes | ranks all-reduce gradients |

With DDP, every rank must execute the same number of backward passes in the same collective order. A rank that runs out of data early can hang every other rank.

In [ ]:
import hashlib
import json
import os
import random
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Iterator

import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, IterableDataset, get_worker_info

torch.manual_seed(7)
torch.set_printoptions(precision=4, sci_mode=False)

In [ ]:
@dataclass
class TrainConfig:
    input_dim: int = 16
    num_classes: int = 4
    batch_size: int = 32
    num_workers: int = 0       # Set > 0 in a script after profiling input cost.
    prefetch_factor: int = 2
    learning_rate: float = 3e-3
    max_steps: int = 40
    checkpoint_every: int = 20
    log_every: int = 5
    seed: int = 1234


def distributed_context() -> tuple[int, int, int]:
    """Return rank, world size, and local rank in local or torchrun mode."""
    if dist.is_available() and dist.is_initialized():
        return dist.get_rank(), dist.get_world_size(), int(os.environ.get("LOCAL_RANK", 0))
    return 0, 1, 0


config = TrainConfig()
rank, world_size, local_rank = distributed_context()
print(asdict(config))
print({"rank": rank, "world_size": world_size, "local_rank": local_rank})

## 2. Deterministic streaming and sharding

An `IterableDataset` is useful when data does not fit locally or arrives as a stream. Each `(rank, worker)` must receive a disjoint shard; otherwise samples are duplicated.

For rank `r` with worker `w`:

```python
global_worker_id = r * workers_per_rank + w
total_consumers = world_size * workers_per_rank
keep sample_id when sample_id % total_consumers == global_worker_id
```

The example generates each record from `sample_id`, so replaying an ID produces identical data. A production source should provide the same property through immutable object names, manifest row IDs, Kafka partition/offset pairs, or another stable cursor.

**Important consistency detail:** a dataset object running in a worker process cannot reliably update checkpoint state in the parent process. DataLoader prefetching also means the source cursor may be ahead of the last completed optimizer step. For exact recovery, checkpoint a committed batch/record ID owned by the trainer, or accept replay and make sample processing idempotent.

In [ ]:
class DeterministicStream(IterableDataset):
    """Infinite synthetic stream whose records are reproducible by sample ID."""

    def __init__(
        self,
        input_dim: int,
        num_classes: int,
        start_sample_id: int = 0,
        seed: int = 0,
        rank: int = 0,
        world_size: int = 1,
    ):
        self.input_dim = input_dim
        self.num_classes = num_classes
        self.start_sample_id = start_sample_id
        self.seed = seed
        self.rank = rank
        self.world_size = world_size

    def _record(self, sample_id: int) -> dict[str, torch.Tensor]:
        generator = torch.Generator().manual_seed(self.seed + sample_id)
        features = torch.randn(self.input_dim, generator=generator)
        # Deterministic target with a learnable signal.
        target = ((features[:4] > 0).sum().long() + sample_id) % self.num_classes
        return {
            "sample_id": torch.tensor(sample_id, dtype=torch.long),
            "features": features,
            "target": target,
        }

    def __iter__(self) -> Iterator[dict[str, torch.Tensor]]:
        worker = get_worker_info()
        worker_id = 0 if worker is None else worker.id
        workers_per_rank = 1 if worker is None else worker.num_workers
        consumer_id = self.rank * workers_per_rank + worker_id
        total_consumers = self.world_size * workers_per_rank

        sample_id = self.start_sample_id + consumer_id
        while True:
            yield self._record(sample_id)
            sample_id += total_consumers

In [ ]:
def seed_worker(worker_id: int) -> None:
    # PyTorch assigns a distinct initial seed to each DataLoader worker.
    worker_seed = torch.initial_seed() % (2**32)
    random.seed(worker_seed)


def make_dataloader(config: TrainConfig, start_sample_id: int = 0) -> DataLoader:
    rank, world_size, _ = distributed_context()
    dataset = DeterministicStream(
        input_dim=config.input_dim,
        num_classes=config.num_classes,
        start_sample_id=start_sample_id,
        seed=config.seed,
        rank=rank,
        world_size=world_size,
    )
    kwargs = {
        "dataset": dataset,
        "batch_size": config.batch_size,
        "num_workers": config.num_workers,
        "pin_memory": torch.cuda.is_available(),
        "worker_init_fn": seed_worker,
        "persistent_workers": config.num_workers > 0,
    }
    if config.num_workers > 0:
        kwargs["prefetch_factor"] = config.prefetch_factor
    return DataLoader(**kwargs)


preview_loader = make_dataloader(config)
preview_batch = next(iter(preview_loader))
print("sample IDs:", preview_batch["sample_id"][:8].tolist())
print("feature shape:", tuple(preview_batch["features"].shape))

### Map-style datasets

For a finite dataset with random access, prefer `DistributedSampler`:

```python
sampler = torch.utils.data.DistributedSampler(
    dataset, num_replicas=world_size, rank=rank, shuffle=True, drop_last=True
)
loader = DataLoader(dataset, sampler=sampler, shuffle=False, ...)

for epoch in range(num_epochs):
    sampler.set_epoch(epoch)  # all ranks use the same new shuffle
```

`drop_last=True` is often used during training so every rank receives the same number of batches. Validation metrics should usually keep all examples and correctly account for sampler padding or use a sampler that does not duplicate records.

## 3. DDP initialization and model construction

DDP normally uses one process per GPU. It broadcasts initial parameters and all-reduces gradients during backward. It does **not** shard the model or optimizer state; use FSDP or another sharded approach when replication no longer fits.

Launch a script with elastic restart support:

```bash
torchrun \
  --nnodes=2 \
  --nproc-per-node=8 \
  --rdzv-backend=c10d \
  --rdzv-endpoint=host0:29400 \
  --max-restarts=3 \
  train.py
```

`torchrun` can restart failed processes, but the application must reload a durable checkpoint. A checkpoint stored only on the failed node is not sufficient.

In [ ]:
def setup_distributed() -> torch.device:
    """Initialize from torchrun environment variables, or stay local."""
    if "RANK" not in os.environ:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")

    local_rank = int(os.environ["LOCAL_RANK"])
    backend = "nccl" if torch.cuda.is_available() else "gloo"
    if torch.cuda.is_available():
        torch.cuda.set_device(local_rank)
    dist.init_process_group(backend=backend)
    return torch.device("cuda", local_rank) if torch.cuda.is_available() else torch.device("cpu")


class TinyClassifier(nn.Module):
    def __init__(self, input_dim: int, num_classes: int):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


def build_model(config: TrainConfig, device: torch.device) -> nn.Module:
    model = TinyClassifier(config.input_dim, config.num_classes).to(device)
    if dist.is_available() and dist.is_initialized():
        device_ids = [device.index] if device.type == "cuda" else None
        model = DDP(model, device_ids=device_ids)
    return model


def unwrapped_model(model: nn.Module) -> nn.Module:
    return model.module if isinstance(model, DDP) else model

## 4. Checkpointing and recovery

A resumable checkpoint should normally include:

- model parameters and buffers
- optimizer and learning-rate scheduler state
- mixed-precision scaler state when using FP16
- completed optimizer step and committed data position
- random-number-generator state
- configuration, code/data version, and schema version

Only rank 0 writes the shared checkpoint. Write a temporary file and atomically rename it so a crash does not expose a partially written checkpoint. Save to durable shared storage and apply retention asynchronously in production.

The example stores `next_sample_id` as a simple local demonstration. At scale, store source-native positions per partition/shard. If world size changes after an elastic restart, resume from a globally meaningful cursor and recompute ownership under the new topology.

In [ ]:
def capture_rng_state() -> dict:
    state = {
        "python": random.getstate(),
        "torch": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state["cuda"] = torch.cuda.get_rng_state_all()
    return state


def restore_rng_state(state: dict) -> None:
    random.setstate(state["python"])
    torch.set_rng_state(state["torch"])
    if torch.cuda.is_available() and "cuda" in state:
        torch.cuda.set_rng_state_all(state["cuda"])


def save_checkpoint(
    path: Path,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler.LRScheduler,
    step: int,
    next_sample_id: int,
    config: TrainConfig,
) -> None:
    rank, world_size, _ = distributed_context()
    local_rng_state = capture_rng_state()
    if dist.is_available() and dist.is_initialized():
        rng_by_rank = [None] * world_size if rank == 0 else None
        dist.gather_object(local_rng_state, rng_by_rank, dst=0)
    else:
        rng_by_rank = [local_rng_state]

    if rank == 0:
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary_path = path.with_suffix(path.suffix + ".tmp")
        payload = {
            "schema_version": 1,
            "model": unwrapped_model(model).state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "step": step,
            "next_sample_id": next_sample_id,
            "rng_by_rank": rng_by_rank,
            "config": asdict(config),
        }
        torch.save(payload, temporary_path)
        os.replace(temporary_path, path)

    if dist.is_available() and dist.is_initialized():
        dist.barrier()


def load_checkpoint(
    path: Path,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler.LRScheduler,
    device: torch.device,
) -> tuple[int, int]:
    if not path.exists():
        return 0, 0

    checkpoint = torch.load(path, map_location=device, weights_only=False)
    unwrapped_model(model).load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    scheduler.load_state_dict(checkpoint["scheduler"])
    rank, world_size, _ = distributed_context()
    saved_rng = checkpoint["rng_by_rank"]
    if len(saved_rng) == world_size:
        restore_rng_state(saved_rng[rank])
    else:
        # Elastic membership changed, so derive distinct streams for the new ranks.
        random.seed(checkpoint["config"]["seed"] + rank)
        torch.manual_seed(checkpoint["config"]["seed"] + rank)
    return int(checkpoint["step"]), int(checkpoint["next_sample_id"])

## 5. Metrics for correctness and debugging

Log reduced global metrics from rank 0, not an arbitrary rank-local value.

| Category | Useful metrics | Typical diagnosis |
|---|---|---|
| optimization | loss, accuracy, learning rate, gradient norm, skipped AMP steps | divergence, bad schedule, overflow |
| data | samples/sec, data wait time, sample-ID range, invalid/duplicate count | loader bottleneck or bad sharding |
| systems | step time, GPU utilization, allocated/reserved memory, checkpoint time | straggler, memory pressure, I/O stall |
| distributed | rank, world size, collective timeout/restart count | failed or desynchronized workers |

Average losses by the number of examples, not by the number of ranks. This matters when local batch sizes differ.

In [ ]:
@torch.no_grad()
def reduce_sums(values: list[float], device: torch.device) -> list[float]:
    tensor = torch.tensor(values, dtype=torch.float64, device=device)
    if dist.is_available() and dist.is_initialized():
        dist.all_reduce(tensor, op=dist.ReduceOp.SUM)
    return tensor.cpu().tolist()


@torch.no_grad()
def reduce_max(value: float, device: torch.device) -> float:
    tensor = torch.tensor(value, dtype=torch.float64, device=device)
    if dist.is_available() and dist.is_initialized():
        dist.all_reduce(tensor, op=dist.ReduceOp.MAX)
    return tensor.item()


def gradient_norm(model: nn.Module) -> float:
    squared_norm = torch.zeros((), device=next(model.parameters()).device)
    for parameter in model.parameters():
        if parameter.grad is not None:
            squared_norm += parameter.grad.detach().float().pow(2).sum()
    return squared_norm.sqrt().item()


def log_metrics(metrics: dict) -> None:
    rank, _, _ = distributed_context()
    if rank == 0:
        print(json.dumps(metrics, sort_keys=True))

## 6. Data consistency checks

Useful invariants to assert or monitor:

1. Every rank performs the same number of optimizer steps.
2. Global batch size is `local_batch * world_size * accumulation_steps` when local batches are full.
3. Sample ownership is disjoint across consumers.
4. A record ID maps to immutable content and preprocessing version.
5. Checkpoint data position advances only after the optimizer step commits.
6. Training and validation never share sample IDs.

For large jobs, log compact fingerprints rather than every ID. A cryptographic hash can detect accidental data-content changes; approximate cardinality sketches or sampled ID audits can help detect duplicates.

In [ ]:
def batch_fingerprint(batch: dict[str, torch.Tensor]) -> str:
    digest = hashlib.sha256()
    digest.update(batch["sample_id"].contiguous().numpy().tobytes())
    digest.update(batch["features"].contiguous().numpy().tobytes())
    digest.update(batch["target"].contiguous().numpy().tobytes())
    return digest.hexdigest()[:16]


first = next(iter(make_dataloader(config)))
replayed = next(iter(make_dataloader(config)))
print("fingerprint:", batch_fingerprint(first))
assert batch_fingerprint(first) == batch_fingerprint(replayed)
assert first["sample_id"].unique().numel() == first["sample_id"].numel()

## 7. End-to-end training loop

The loop measures data wait separately from compute time, reduces weighted metrics, clips and logs gradient norm, and checkpoints only after a completed optimizer step.

In a true streaming system, replace the simple `next_sample_id` calculation with committed source offsets. Exactly-once training is rarely free: most systems choose **at-least-once replay** after failure because deterministic, idempotent replay is simpler than distributed transactions between the trainer, checkpoint store, and data source.

In [ ]:
def train(
    config: TrainConfig,
    checkpoint_path: Path,
    stop_after_step: int | None = None,
) -> nn.Module:
    device = setup_distributed()
    rank, world_size, _ = distributed_context()

    # Same initialization seed on every rank; DDP also broadcasts rank 0 state.
    torch.manual_seed(config.seed)
    model = build_model(config, device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=config.max_steps
    )

    start_step, next_sample_id = load_checkpoint(
        checkpoint_path, model, optimizer, scheduler, device
    )
    loader = make_dataloader(config, start_sample_id=next_sample_id)
    iterator = iter(loader)
    model.train()

    previous_step_end = time.perf_counter()
    final_step = config.max_steps if stop_after_step is None else stop_after_step
    for step in range(start_step, min(final_step, config.max_steps)):
        batch = next(iterator)
        data_time = time.perf_counter() - previous_step_end
        step_start = time.perf_counter()

        features = batch["features"].to(device, non_blocking=True)
        targets = batch["target"].to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        logits = model(features)
        loss = F.cross_entropy(logits, targets)
        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0).item()
        optimizer.step()
        scheduler.step()

        correct = (logits.argmax(dim=-1) == targets).sum().item()
        local_count = targets.numel()
        loss_sum, correct_sum, global_count = reduce_sums(
            [loss.item() * local_count, correct, local_count], device
        )
        step_time = time.perf_counter() - step_start
        max_data_time = reduce_max(data_time, device)
        max_step_time = reduce_max(step_time, device)

        # This cursor is illustrative because the synthetic stream has one global order.
        next_sample_id += int(global_count)

        if (step + 1) % config.log_every == 0:
            metrics = {
                "step": step + 1,
                "loss": loss_sum / global_count,
                "accuracy": correct_sum / global_count,
                "learning_rate": optimizer.param_groups[0]["lr"],
                "gradient_norm": grad_norm,
                "max_data_time_s": max_data_time,
                "max_step_time_s": max_step_time,
                "samples_per_s": global_count / max(max_step_time, 1e-12),
                "next_sample_id": next_sample_id,
                "world_size": world_size,
            }
            if device.type == "cuda":
                metrics["cuda_memory_allocated_mb"] = (
                    torch.cuda.memory_allocated(device) / 2**20
                )
            log_metrics(metrics)

        if (step + 1) % config.checkpoint_every == 0:
            save_checkpoint(
                checkpoint_path,
                model,
                optimizer,
                scheduler,
                step=step + 1,
                next_sample_id=next_sample_id,
                config=config,
            )

        previous_step_end = time.perf_counter()

    if dist.is_available() and dist.is_initialized():
        dist.destroy_process_group()
    return unwrapped_model(model).cpu()


demo_config = TrainConfig(max_steps=12, checkpoint_every=2, log_every=2)
demo_checkpoint = Path("/tmp/distributed-training-demo.pt")
demo_checkpoint.unlink(missing_ok=True)
# Simulate a job stopping after its step-10 checkpoint.
trained_model = train(demo_config, demo_checkpoint, stop_after_step=10)
print("checkpoint exists:", demo_checkpoint.exists())

In [ ]:
# Resume from step 10 and continue with the identical training configuration.
resumed_model = train(demo_config, demo_checkpoint)
saved = torch.load(demo_checkpoint, map_location="cpu", weights_only=False)
print("resumed checkpoint step:", saved["step"])
print("resumed next sample ID:", saved["next_sample_id"])
demo_checkpoint.unlink(missing_ok=True)

## 8. Failure scenarios

### DataLoader worker fails

The parent rank usually raises an exception. Let the training process fail so the job supervisor can restart it from a checkpoint. Silently skipping a corrupt record can desynchronize ranks unless every rank still executes the same number of steps. Log the stable record ID and quarantine policy.

### One DDP rank or node fails

Other ranks commonly block or receive a collective error. They cannot safely continue with stale membership. Elastic `torchrun` terminates and recreates the worker group; every rank reloads the same durable checkpoint. With membership changes, repartition the input stream from globally committed offsets.

### Rank 0 fails while checkpointing

Atomic rename prevents readers from seeing a half-written local file. Durable object storage may not provide filesystem rename semantics, so upload a uniquely named immutable checkpoint and update a small `LATEST` manifest only after upload validation.

### Failure after optimizer step but before checkpoint

On restart, the job replays work since the previous checkpoint. This is at-least-once processing. Deterministic data and RNG restoration make replay predictable, although bitwise reproducibility can still depend on kernels, hardware, and collective ordering.

## 9. Interview prompts and concise answers

**Why DDP instead of `DataParallel`?**  
DDP uses one process per accelerator, avoids a single Python process bottleneck, and overlaps gradient communication with backward computation.

**Why call `DistributedSampler.set_epoch(epoch)`?**  
It changes the deterministic shuffle each epoch while preserving the same ordering basis across ranks.

**Can each rank save its own checkpoint?**  
For ordinary DDP, model states are replicas, so rank 0 can save one full checkpoint. Sharded training may require coordinated per-rank shards plus metadata.

**Does restoring the model alone resume training?**  
No. Missing optimizer, scheduler, scaler, RNG, and data position changes the optimization trajectory and can replay or skip data.

**How do you prevent duplicate streamed samples?**  
Assign stable source partitions or deterministically shard stable IDs across rank-worker consumers. Audit IDs or fingerprints and define replay semantics explicitly.

**What causes a distributed hang?**  
Common causes are unequal batch counts, conditional backward/collectives, a failed rank, blocked input I/O, or mismatched process-group configuration.

**What would you inspect when GPUs are underutilized?**  
Compare data wait and compute time, profile decoding/augmentation, tune worker count and prefetching, inspect host-to-device copies, batch size, stragglers, and collective time.

**What changes for very large models?**  
DDP replicates all state. Use FSDP/ZeRO for parameter, gradient, and optimizer-state sharding; tensor or pipeline parallelism may also be required.

## 10. Production checklist

- [ ] one training process per GPU and explicit rank/device mapping
- [ ] deterministic, disjoint data ownership across ranks and loader workers
- [ ] equal collective order and optimizer-step count on every rank
- [ ] durable atomic/versioned checkpoints with full training state
- [ ] explicit at-least-once or exactly-once data semantics
- [ ] elastic launcher or scheduler restart policy tested with injected failures
- [ ] globally reduced metrics plus rank-local diagnostic logs
- [ ] data time, compute time, throughput, gradient norm, LR, and memory monitored
- [ ] dataset, preprocessing, configuration, and code versions recorded
- [ ] checkpoint restore and data replay tested before starting an expensive run